In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/modma-dataset-2/audio_lanzhou_2015/audio_lanzhou_2015/MODMA dataset-a Multi-modal Open Dataset for Mental-disorder Analysis.pdf
/kaggle/input/modma-dataset-2/audio_lanzhou_2015/audio_lanzhou_2015/subjects_information_audio_lanzhou_2015.xlsx
/kaggle/input/modma-dataset-2/audio_lanzhou_2015/audio_lanzhou_2015/A Novel Decision Tree for Depression Recognition in Speech.pdf
/kaggle/input/modma-dataset-2/audio_lanzhou_2015/audio_lanzhou_2015/02020008/17.wav
/kaggle/input/modma-dataset-2/audio_lanzhou_2015/audio_lanzhou_2015/02020008/10.wav
/kaggle/input/modma-dataset-2/audio_lanzhou_2015/audio_lanzhou_2015/02020008/14.wav
/kaggle/input/modma-dataset-2/audio_lanzhou_2015/audio_lanzhou_2015/02020008/02.wav
/kaggle/input/modma-dataset-2/audio_lanzhou_2015/audio_lanzhou_2015/02020008/03.wav
/kaggle/input/modma-dataset-2/audio_lanzhou_2015/audio_lanzhou_2015/02020008/05.wav
/kaggle/input/modma-dataset-2/audio_lanzhou_2015/audio_lanzhou_2015/02020008/19.wav
/kaggle/input/modma-datase

# **MAT-128-CHANNEL-PREPROCESSING**

In [2]:
import os
import mne
import scipy.io
import numpy as np

# Define input and output directory paths
input_dir = '/kaggle/input/modma-dataset-2/EEG_128channels_resting_lanzhou_2015/EEG_128channels_resting_lanzhou_2015'
output_dir = '/kaggle/working/processed_mat_data2/'  # Define your desired output directory

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Iterate through each file in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.mat'):
        # Load mat file
        mat_data = scipy.io.loadmat(os.path.join(input_dir, filename))
        
        # Print keys in mat_data for inspection
        print(f"Keys in {filename}: {mat_data.keys()}")
        
        # Example: Extract EEG data from the first suitable key found
        eeg_data = None
        eeg_data_key = None
        
        for key in mat_data.keys():
            if isinstance(mat_data[key], (np.ndarray, list)) and len(mat_data[key]) > 0:
                if isinstance(mat_data[key][0], (np.ndarray, list)) and len(mat_data[key][0]) > 0:
                    eeg_data = mat_data[key]
                    eeg_data_key = key
                    break
        
        if eeg_data is None:
            print(f"No suitable EEG data found in {filename}. Skipping...")
            continue
        
        # Assuming sampling rate is stored in 'samplingRate' key
        sampling_rate = float(mat_data['samplingRate'][0, 0])
        
        # Ensure EEG data is in the expected format (channels x samples)
        eeg_data = np.array(eeg_data).T  # Transpose if necessary
        
        # Determine number of channels from EEG data shape
        n_channels = eeg_data.shape[0]
        
        # Create channel names based on the number of channels
        ch_names = [f'EEG {i+1:03}' for i in range(n_channels)]
        
        # Create MNE info structure
        info = mne.create_info(ch_names=ch_names, sfreq=sampling_rate, ch_types='eeg')
        
        # Create RawArray from EEG data and info
        raw = mne.io.RawArray(eeg_data, info)
        
        # Apply preprocessing steps (example: filtering)
        raw.filter(0.5, 45, fir_design='firwin')
        
        # Save processed data to the output directory with .fif extension
        processed_filename = os.path.join(output_dir, f'processed_{os.path.splitext(filename)[0]}.fif')
        raw.save(processed_filename, overwrite=True)


Keys in 02030007_rest 20151103 2032.mat: dict_keys(['__header__', '__version__', '__globals__', 'a02030007_rest_20151103_2032mat', 'samplingRate', 'Impedances_0'])
Creating RawArray with float64 data, n_channels=75750, n_times=129
    Range : 0 ... 128 =      0.000 ...     0.512 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 1651 samples (6.604 s)



/tmp/ipykernel_33/2653124871.py:56: RuntimeWarning: filter_length (1651) is longer than the signal (129), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(0.5, 45, fir_design='firwin')
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 647 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 881 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 1151 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 1457 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 1799 tasks      | elapsed:    0.4s
[Parallel(n_jobs=1)]: Done 2177 tasks      | elapsed:    0.5s
[Parallel(n_jobs=1)]: Done 2591 tasks      | elapsed:    0.6s
[Parallel(n_jobs=1)]: Done 3041 tasks      |

Writing /kaggle/working/processed_mat_data2/processed_02030007_rest 20151103 2032.fif
Closing /kaggle/working/processed_mat_data2/processed_02030007_rest 20151103 2032.fif
[done]
Keys in 02030003_rest 20151022 1155.mat: dict_keys(['__header__', '__version__', '__globals__', 'a02030003_rest_20151022_1155mat', 'samplingRate', 'Impedances_0'])
Creating RawArray with float64 data, n_channels=75513, n_times=129
    Range : 0 ... 128 =      0.000 ...     0.512 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filt

/tmp/ipykernel_33/2653124871.py:56: RuntimeWarning: filter_length (1651) is longer than the signal (129), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(0.5, 45, fir_design='firwin')
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 647 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 881 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 1151 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 1457 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 1799 tasks      | elapsed:    0.4s
[Parallel(n_jobs=1)]: Done 2177 tasks      | elapsed:    0.5s
[Parallel(n_jobs=1)]: Done 2591 tasks      | elapsed:    0.6s
[Parallel(n_jobs=1)]: Done 3041 tasks      |

Writing /kaggle/working/processed_mat_data2/processed_02030003_rest 20151022 1155.fif
Closing /kaggle/working/processed_mat_data2/processed_02030003_rest 20151022 1155.fif
[done]
Keys in 02020010rest 20150625 1224..mat: dict_keys(['__header__', '__version__', '__globals__', 'a02020010rest_20150625_1224mat', 'samplingRate', 'Impedances_0'])
Creating RawArray with float64 data, n_channels=76039, n_times=129
    Range : 0 ... 128 =      0.000 ...     0.512 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filte

...

Writing /kaggle/working/processed_mat_data2/processed_02020016rest 20150701 1040..fif
Closing /kaggle/working/processed_mat_data2/processed_02020016rest 20150701 1040..fif
[done]
Keys in 02020022rest 20150707 1452..mat: dict_keys(['__header__', '__version__', '__globals__', 'a02020022rest_20150707_1452mat', 'samplingRate', 'Impedances_0'])
Creating RawArray with float64 data, n_channels=75139, n_times=129
    Range : 0 ... 128 =      0.000 ...     0.512 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filte

/tmp/ipykernel_33/2653124871.py:56: RuntimeWarning: filter_length (1651) is longer than the signal (129), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(0.5, 45, fir_design='firwin')
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 647 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 881 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 1151 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 1457 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 1799 tasks      | elapsed:    0.4s
[Parallel(n_jobs=1)]: Done 2177 tasks      | elapsed:    0.5s
[Parallel(n_jobs=1)]: Done 2591 tasks      | elapsed:    0.6s
[Parallel(n_jobs=1)]: Done 3041 tasks      |

Writing /kaggle/working/processed_mat_data2/processed_02020022rest 20150707 1452..fif
Closing /kaggle/working/processed_mat_data2/processed_02020022rest 20150707 1452..fif
[done]
Keys in 02020008rest 20150624 1711..mat: dict_keys(['__header__', '__version__', '__globals__', 'a02020008rest_20150624_1711mat', 'samplingRate', 'Impedances_0'])
Creating RawArray with float64 data, n_channels=75101, n_times=129
    Range : 0 ... 128 =      0.000 ...     0.512 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filte

/tmp/ipykernel_33/2653124871.py:56: RuntimeWarning: filter_length (1651) is longer than the signal (129), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(0.5, 45, fir_design='firwin')
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 647 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 881 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 1151 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 1457 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 1799 tasks      | elapsed:    0.4s
[Parallel(n_jobs=1)]: Done 2177 tasks      | elapsed:    0.5s
[Parallel(n_jobs=1)]: Done 2591 tasks      | elapsed:    0.6s
[Parallel(n_jobs=1)]: Done 3041 tasks      |

Writing /kaggle/working/processed_mat_data2/processed_02020008rest 20150624 1711..fif
Closing /kaggle/working/processed_mat_data2/processed_02020008rest 20150624 1711..fif
[done]


In [5]:
import os
import mne
import scipy.io
import numpy as np

# Define input and output directory paths
input_dir = '/kaggle/input/modma-dataset-2/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'
output_dir = '/kaggle/working/processed_raw_data/'  # Define your desired output directory

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Iterate through each file in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.mat'):
        # Load mat file
        mat_data = scipy.io.loadmat(os.path.join(input_dir, filename))
        
        # Print keys in mat_data for inspection
        print(f"Keys in {filename}: {mat_data.keys()}")
        
        # Example: Extract EEG data from the first suitable key found
        eeg_data = None
        eeg_data_key = None
        
        for key in mat_data.keys():
            if isinstance(mat_data[key], (np.ndarray, list)) and len(mat_data[key]) > 0:
                if isinstance(mat_data[key][0], (np.ndarray, list)) and len(mat_data[key][0]) > 0:
                    eeg_data = mat_data[key]
                    eeg_data_key = key
                    break
        
        if eeg_data is None:
            print(f"No suitable EEG data found in {filename}. Skipping...")
            continue
        
        # Print the structure of 'chaninfo' to inspect how to correctly extract the sampling rate
        print(f"Structure of 'chaninfo' in {filename}: {mat_data['chaninfo']}")
        
        # Correctly extract the sampling rate from 'chaninfo'
        sampling_rate = float(mat_data['chaninfo']['sampling_rate'][0, 0])
        
        # Ensure EEG data is in the expected format (channels x samples)
        eeg_data = np.array(eeg_data).T  # Transpose if necessary
        
        # Determine number of channels from EEG data shape
        n_channels = eeg_data.shape[0]
        
        # Create channel names based on the number of channels
        ch_names = [f'EEG {i+1:03}' for i in range(n_channels)]
        
        # Create MNE info structure
        info = mne.create_info(ch_names=ch_names, sfreq=sampling_rate, ch_types='eeg')
        
        # Create RawArray from EEG data and info
        raw = mne.io.RawArray(eeg_data, info)
        
        # Apply preprocessing steps (example: filtering)
        raw.filter(0.5, 45, fir_design='firwin')
        
        # Save processed data to the output directory with .fif extension
        processed_filename = os.path.join(output_dir, f'processed_{os.path.splitext(filename)[0]}.fif')
        raw.save(processed_filename, overwrite=True)


Keys in chan_info_egi_128.mat: dict_keys(['__header__', '__version__', '__globals__', 'chaninfo', 'chanlocs'])
Structure of 'chaninfo' in chan_info_egi_128.mat: [[(array(['D:\\motor image data\\EGI_GSN_HydroCel_128.sfp'], dtype='<U44'), array(['E1\t5.787677636\t5.520863216\t-2.577468644   ',
         'E2\t5.291804727\t6.709097557\t0.307434896    ',
         'E3\t3.864122447\t7.63424051\t3.067770143     ',
         'E4\t2.868837559\t7.145708546\t4.989564557    ',
         'E5\t1.479340453\t5.68662139\t6.812878187     ',
         'E6\t0\t3.806770224\t7.891304964              ',
         'E7\t-1.223800252\t1.558864431\t8.44043914    ',
         'E8\t4.221901505\t7.998817387\t-1.354789681   ',
         'E9\t2.695405558\t8.884820317\t1.088308144    ',
         'E10\t1.830882336\t8.708839134\t3.18709115    ',
         'E11\t0\t7.96264703\t5.044718001              ',
         'E12\t-1.479340453\t5.68662139\t6.812878187   ',
         'E13\t-2.435870762\t3.254307219\t7.608766206  ',
         'E

ValueError: no field of name sampling_rate

In [6]:
import os
import mne
import scipy.io
import numpy as np

# Define input and output directory paths
input_dir = '/kaggle/input/modma-dataset-2/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'
output_dir = '/kaggle/working/processed_raw_data/'  # Define your desired output directory

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Iterate through each file in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.mat'):
        # Load mat file
        mat_data = scipy.io.loadmat(os.path.join(input_dir, filename))
        
        # Print keys in mat_data for inspection
        print(f"Keys in {filename}: {mat_data.keys()}")
        
        # Example: Extract EEG data from the first suitable key found
        eeg_data = None
        eeg_data_key = None
        
        for key in mat_data.keys():
            if isinstance(mat_data[key], (np.ndarray, list)) and len(mat_data[key]) > 0:
                if isinstance(mat_data[key][0], (np.ndarray, list)) and len(mat_data[key][0]) > 0:
                    eeg_data = mat_data[key]
                    eeg_data_key = key
                    break
        
        if eeg_data is None:
            print(f"No suitable EEG data found in {filename}. Skipping...")
            continue
        
        # Print the structure of 'chaninfo' to inspect how to correctly extract the sampling rate
        print(f"Structure of 'chaninfo' in {filename}: {mat_data['chaninfo']}")
        print(f"Detailed view of 'chaninfo': {mat_data['chaninfo'].dtype}")
        
        # Attempt to extract the sampling rate
        sampling_rate = None
        try:
            sampling_rate = float(mat_data['chaninfo']['samplingRate'][0, 0])
        except KeyError:
            try:
                # If samplingRate is not present, try alternative field names
                sampling_rate = float(mat_data['chaninfo']['sampling_rate'][0, 0])
            except KeyError:
                print(f"Unable to find the sampling rate in 'chaninfo' for {filename}. Skipping...")
                continue
        
        # Ensure EEG data is in the expected format (channels x samples)
        eeg_data = np.array(eeg_data).T  # Transpose if necessary
        
        # Determine number of channels from EEG data shape
        n_channels = eeg_data.shape[0]
        
        # Create channel names based on the number of channels
        ch_names = [f'EEG {i+1:03}' for i in range(n_channels)]
        
        # Create MNE info structure
        info = mne.create_info(ch_names=ch_names, sfreq=sampling_rate, ch_types='eeg')
        
        # Create RawArray from EEG data and info
        raw = mne.io.RawArray(eeg_data, info)
        
        # Apply preprocessing steps (example: filtering)
        raw.filter(0.5, 45, fir_design='firwin')
        
        # Save processed data to the output directory with .fif extension
        processed_filename = os.path.join(output_dir, f'processed_{os.path.splitext(filename)[0]}.fif')
        raw.save(processed_filename, overwrite=True)


Keys in chan_info_egi_128.mat: dict_keys(['__header__', '__version__', '__globals__', 'chaninfo', 'chanlocs'])
Structure of 'chaninfo' in chan_info_egi_128.mat: [[(array(['D:\\motor image data\\EGI_GSN_HydroCel_128.sfp'], dtype='<U44'), array(['E1\t5.787677636\t5.520863216\t-2.577468644   ',
         'E2\t5.291804727\t6.709097557\t0.307434896    ',
         'E3\t3.864122447\t7.63424051\t3.067770143     ',
         'E4\t2.868837559\t7.145708546\t4.989564557    ',
         'E5\t1.479340453\t5.68662139\t6.812878187     ',
         'E6\t0\t3.806770224\t7.891304964              ',
         'E7\t-1.223800252\t1.558864431\t8.44043914    ',
         'E8\t4.221901505\t7.998817387\t-1.354789681   ',
         'E9\t2.695405558\t8.884820317\t1.088308144    ',
         'E10\t1.830882336\t8.708839134\t3.18709115    ',
         'E11\t0\t7.96264703\t5.044718001              ',
         'E12\t-1.479340453\t5.68662139\t6.812878187   ',
         'E13\t-2.435870762\t3.254307219\t7.608766206  ',
         'E

ValueError: no field of name samplingRate

In [8]:
import os
import mne
import scipy.io
import numpy as np

# Define input and output directory paths
input_dir = '/kaggle/input/modma-dataset-2/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'
output_dir = '/kaggle/working/processed_raw_data/'  # Define your desired output directory

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define a default sampling rate if it's not found in the .mat file
default_sampling_rate = 250.0

# Iterate through each file in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.mat'):
        # Load mat file
        mat_data = scipy.io.loadmat(os.path.join(input_dir, filename))
        
        # Print keys in mat_data for inspection
        print(f"Keys in {filename}: {mat_data.keys()}")
        
        # Example: Extract EEG data from the first suitable key found
        eeg_data = None
        eeg_data_key = None
        
        for key in mat_data.keys():
            if isinstance(mat_data[key], (np.ndarray, list)) and len(mat_data[key]) > 0:
                if isinstance(mat_data[key][0], (np.ndarray, list)) and len(mat_data[key][0]) > 0:
                    eeg_data = mat_data[key]
                    eeg_data_key = key
                    break
        
        if eeg_data is None:
            print(f"No suitable EEG data found in {filename}. Skipping...")
            continue
        
        # Assuming sampling rate is stored in 'samplingRate' key
        sampling_rate = default_sampling_rate
        
        # Ensure EEG data is in the expected format (channels x samples)
        eeg_data = np.array(eeg_data).T  # Transpose if necessary
        
        # Determine number of channels from EEG data shape
        n_channels = eeg_data.shape[0]
        
        # Create channel names based on the number of channels
        ch_names = [f'EEG {i+1:03}' for i in range(n_channels)]
        
        # Create MNE info structure
        info = mne.create_info(ch_names=ch_names, sfreq=sampling_rate, ch_types='eeg')
        
        # Create RawArray from EEG data and info
        raw = mne.io.RawArray(eeg_data, info)
        
        # Apply preprocessing steps (example: filtering)
        raw.filter(0.5, 45, fir_design='firwin')
        
        # Save processed data to the output directory with .fif extension
        processed_filename = os.path.join(output_dir, f'processed_{os.path.splitext(filename)[0]}.fif')
        raw.save(processed_filename, overwrite=True)


Keys in chan_info_egi_128.mat: dict_keys(['__header__', '__version__', '__globals__', 'chaninfo', 'chanlocs'])


TypeError: Cannot cast array data from dtype([('filename', 'O'), ('filecontent', 'O'), ('nodatchans', 'O'), ('icachansind', 'O')]) to dtype('float64') according to the rule 'unsafe'

In [9]:
import os
import mne
import scipy.io
import numpy as np

# Define input and output directory paths
input_dir = '/kaggle/input/modma-dataset-2/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'
output_dir = '/kaggle/working/processed_raw_data/'  # Define your desired output directory

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define a default sampling rate if it's not found in the .mat file
default_sampling_rate = 250.0

# Iterate through each file in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.mat'):
        # Load mat file
        mat_data = scipy.io.loadmat(os.path.join(input_dir, filename))
        
        # Print keys in mat_data for inspection
        print(f"Keys in {filename}: {mat_data.keys()}")
        
        # Inspect the structure of 'chaninfo' and 'chanlocs'
        print(f"Structure of 'chaninfo' in {filename}: {mat_data['chaninfo']}")
        print(f"Structure of 'chanlocs' in {filename}: {mat_data['chanlocs']}")
        
        # Example: Extract EEG data from 'chanlocs' or other appropriate key
        eeg_data = None
        eeg_data_key = None
        
        # Iterate through keys to find appropriate EEG data
        for key in mat_data.keys():
            if isinstance(mat_data[key], np.ndarray) and mat_data[key].dtype == np.float64:
                eeg_data = mat_data[key]
                eeg_data_key = key
                break
        
        if eeg_data is None:
            print(f"No suitable EEG data found in {filename}. Skipping...")
            continue
        
        # Assuming sampling rate is stored in 'chaninfo' key or using default
        sampling_rate = default_sampling_rate
        
        if 'chaninfo' in mat_data:
            chaninfo = mat_data['chaninfo']
            # If 'chaninfo' contains the sampling rate, extract it here
        
        # Ensure EEG data is in the expected format (channels x samples)
        eeg_data = np.array(eeg_data).T  # Transpose if necessary
        
        # Determine number of channels from EEG data shape
        n_channels = eeg_data.shape[0]
        
        # Create channel names based on the number of channels
        ch_names = [f'EEG {i+1:03}' for i in range(n_channels)]
        
        # Create MNE info structure
        info = mne.create_info(ch_names=ch_names, sfreq=sampling_rate, ch_types='eeg')
        
        # Create RawArray from EEG data and info
        raw = mne.io.RawArray(eeg_data, info)
        
        # Apply preprocessing steps (example: filtering)
        raw.filter(0.5, 45, fir_design='firwin')
        
        # Save processed data to the output directory with .fif extension
        processed_filename = os.path.join(output_dir, f'processed_{os.path.splitext(filename)[0]}.fif')
        raw.save(processed_filename, overwrite=True)


Keys in chan_info_egi_128.mat: dict_keys(['__header__', '__version__', '__globals__', 'chaninfo', 'chanlocs'])
Structure of 'chaninfo' in chan_info_egi_128.mat: [[(array(['D:\\motor image data\\EGI_GSN_HydroCel_128.sfp'], dtype='<U44'), array(['E1\t5.787677636\t5.520863216\t-2.577468644   ',
         'E2\t5.291804727\t6.709097557\t0.307434896    ',
         'E3\t3.864122447\t7.63424051\t3.067770143     ',
         'E4\t2.868837559\t7.145708546\t4.989564557    ',
         'E5\t1.479340453\t5.68662139\t6.812878187     ',
         'E6\t0\t3.806770224\t7.891304964              ',
         'E7\t-1.223800252\t1.558864431\t8.44043914    ',
         'E8\t4.221901505\t7.998817387\t-1.354789681   ',
         'E9\t2.695405558\t8.884820317\t1.088308144    ',
         'E10\t1.830882336\t8.708839134\t3.18709115    ',
         'E11\t0\t7.96264703\t5.044718001              ',
         'E12\t-1.479340453\t5.68662139\t6.812878187   ',
         'E13\t-2.435870762\t3.254307219\t7.608766206  ',
         'E

In [10]:
import os
import scipy.io
import numpy as np

# Define input directory path
input_dir = '/kaggle/input/modma-dataset-2/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'

# Function to inspect the .mat file contents
def inspect_mat_file(file_path):
    mat_data = scipy.io.loadmat(file_path)
    
    print(f"Keys in {file_path}: {list(mat_data.keys())}")
    
    # Inspecting each key
    for key in mat_data.keys():
        if not key.startswith('__'):
            print(f"\nKey: {key}")
            print(f"Type: {type(mat_data[key])}")
            if isinstance(mat_data[key], np.ndarray):
                print(f"Shape: {mat_data[key].shape}")
                if mat_data[key].size < 100:  # Print content for small arrays for better inspection
                    print(mat_data[key])

# Iterate through each file in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.mat'):
        file_path = os.path.join(input_dir, filename)
        inspect_mat_file(file_path)


Keys in /kaggle/input/modma-dataset-2/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015/chan_info_egi_128.mat: ['__header__', '__version__', '__globals__', 'chaninfo', 'chanlocs']

Key: chaninfo
Type: <class 'numpy.ndarray'>
Shape: (1, 1)
[[(array(['D:\\motor image data\\EGI_GSN_HydroCel_128.sfp'], dtype='<U44'), array(['E1\t5.787677636\t5.520863216\t-2.577468644   ',
         'E2\t5.291804727\t6.709097557\t0.307434896    ',
         'E3\t3.864122447\t7.63424051\t3.067770143     ',
         'E4\t2.868837559\t7.145708546\t4.989564557    ',
         'E5\t1.479340453\t5.68662139\t6.812878187     ',
         'E6\t0\t3.806770224\t7.891304964              ',
         'E7\t-1.223800252\t1.558864431\t8.44043914    ',
         'E8\t4.221901505\t7.998817387\t-1.354789681   ',
         'E9\t2.695405558\t8.884820317\t1.088308144    ',
         'E10\t1.830882336\t8.708839134\t3.18709115    ',
         'E11\t0\t7.96264703\t5.044718001              ',
         'E12\t-1.479340453\t5.68

In [1]:
!zip -r mat-128-channel-preprocessed-output.zip /kaggle/working/processed_mat_data2

  adding: kaggle/working/processed_mat_data2/ (stored 0%)
  adding: kaggle/working/processed_mat_data2/processed_02020019rest 20150703 1036..fif (deflated 36%)
  adding: kaggle/working/processed_mat_data2/processed_02030020_rest 20151230 1416.fif (deflated 29%)
  adding: kaggle/working/processed_mat_data2/processed_02020029rest 20150715 1316..fif (deflated 40%)
  adding: kaggle/working/processed_mat_data2/processed_02020010rest 20150625 1224..fif (deflated 39%)
  adding: kaggle/working/processed_mat_data2/processed_02010004rest 20150427 1335..fif (deflated 36%)
  adding: kaggle/working/processed_mat_data2/processed_02010002rest 20150416 1017..fif (deflated 35%)
  adding: kaggle/working/processed_mat_data2/processed_02020020rest 20150703 1754..fif (deflated 35%)
  adding: kaggle/working/processed_mat_data2/processed_02010019rest 20150716 1440..fif (deflated 35%)
  adding: kaggle/working/processed_mat_data2/processed_02010015rest 20150709 1456..fif (deflated 33%)
  adding: kaggle/working

In [1]:
!ls

mat-128-channel-preprocessed-output.zip  processed_mat_data2  state.db


In [1]:
from IPython.display import FileLink
FileLink(r'mat-128-channel-preprocessed-output.zip')

/kaggle/working/mat-128-channel-preprocessed-output.zip